# All of Us — AKI → CKD phenotype from OMOP (prototype)

**Author:** hkgoren. Developed and validated on a mock CDR (`mock/aou_mock.py`); runs unchanged on the All of Us Researcher Workbench, where `WORKSPACE_CDR` is set and `run_sql` goes to BigQuery.

> **If you are an AI assistant running this on the Workbench:** the functions and clinical decisions in this notebook are **fixed**. Do not modify, "improve" or rewrite them. Allowed: adding `LIMIT` to SQL for a dry run, debugging, reporting runtimes. Full rules in `docs/WORKBENCH_INSTRUCTIONS.md`.

## Pipeline

1. SQL: creatinine measurements, person table, inpatient stays (9201 inpatient, 262 ER → inpatient)
2. Sex and age → measurement table
3. Unit codes → labels (`mg/dL`, `umol/L`)
4. `assign_stays`: tag each measurement with a `stay_no` (NaN if outpatient) → split inpatient / outpatient
5. `detect_aki`: per stay — umol→mg/dL, drop >10 mg/dL, AKI flag, months since peak creatinine
6. `convert_units`: umol→mg/dL for outpatient rows
7. `calculate_egfr`: CKD-EPI 2021 (race-free)
8. `compute_time_since_each_aki`: months since each AKI peak, for every row of the person
9. `detect_ckd`: per person — baseline vs follow-up eGFR → `ckd_developed`, `drop_flag`

Output: **`aki_ckd_table`** (one row per creatinine measurement; person-level flags written on every row).

## 0. Setup

In [1]:
%pip -q install duckdb sqlglot pandas

In [2]:
from datetime import datetime, timedelta

In [3]:
# Colab: if aou_mock.py lives in the all_of_us folder on Drive
import sys, os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    sys.path.insert(0, "/content/drive/MyDrive/all_of_us")   # adjust to where aou_mock.py lives on your Drive
except ImportError:
    sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..', 'mock'))
    pass            # not Colab: aou_mock.py must sit next to this notebook (../mock)

import pandas as pd
import numpy as np
import aou_mock
from aou_mock import run_sql, dry_run, columns, CDR

con = aou_mock.connect()   # builds the mock CDR locally; no-op on the Workbench
print("CDR =", CDR, "| on Workbench:", aou_mock.in_workbench())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
CDR = aou-cdr-mock.C2025Q4R6 | on Workbench: False


## 1. SQL: concept vocabulary, creatinine, person, stays

* `concept`: to map unit codes (`mg/dL`, `umol/L`) to concept ids.
* `measurement`: the two creatinine concepts — 3016723 (mass, mg/dL) and 3020564 (molar). `unit_concept_id` is left raw; cleaning happens in pandas.
* `person`: `sex_at_birth_source_value`, `year_of_birth`.
* `visit_occurrence`: real inpatient stays only. 9202 (outpatient) and 9203 (ER only) are **not** stays.

In [4]:
concept = run_sql(f'''
SELECT concept_id, concept_name, domain_id, vocabulary_id, standard_concept, concept_code, concept_name,
FROM `{CDR}.concept`
ORDER BY concept_id
''')

In [5]:
measurement = run_sql(f'''
SELECT measurement_id, person_id, measurement_date, measurement_datetime, value_as_number, measurement_concept_id, unit_concept_id, unit_source_value
FROM `{CDR}.measurement`
WHERE measurement_concept_id IN (3016723, 3020564)
ORDER BY measurement_concept_id
''')

In [6]:
person = run_sql(f'''
SELECT *,
FROM `{CDR}.person`
''')

In [7]:
stays = run_sql(f'''
SELECT *
FROM `{CDR}.visit_occurrence`
WHERE visit_concept_id in (9201,262)
''')

## 2. Sex and age

* `sex`: 1 male, 0 female, anything else NaN (CKD-EPI has two sex coefficients; NaN rows get no eGFR).
* `age_at_measurement`: measurement year − birth year. All of Us gives no birthday, so age is year-level.

In [8]:
sex_map = {'SexAtBirth_Male':1,
                    'SexAtBirth_Female':0}

In [9]:
person['sex'] = pd.to_numeric(person['sex_at_birth_source_value'].replace(sex_map),errors = 'coerce')
person_to_merge = person.loc[:, ['person_id','sex', 'year_of_birth']]

In [10]:
person_id_sex_map = dict(zip(person_to_merge['person_id'], person_to_merge['sex']))
person_id_birth_year = dict(zip(person_to_merge['person_id'], person_to_merge['year_of_birth']))

In [11]:
measurement['sex'] = measurement['person_id'].map(person_id_sex_map)
measurement['birth_year'] = measurement['person_id'].map(person_id_birth_year)

In [12]:
measurement['age_at_measurement']  = measurement['measurement_date'].dt.year - measurement['birth_year']

## 3. Unit codes → labels

`get_unit_vocab` finds the concept ids for `mg/dL` and `umol/L` in the concept table; those codes in `unit_concept_id` are replaced by the labels. Other codes (0, NULL, 720870, …) stay as strings; downstream, anything not containing `mol` is treated as mg/dL.

In [13]:
def get_unit_vocab(requested_units = list):
  '''
  Looks up the requested unit codes in the concept table and returns
  a {concept_id (str): concept_code} map for use with .replace().
  '''
  keys = []
  for unit in requested_units:
    try:
      key = concept[concept['concept_code'].str.contains(unit, case = False)]['concept_id'].values[0]
      keys.append(key)
    except:
      print(f"{unit}: no matching unit concept found.")

  unit_vocab = {}
  for key in keys:
    unit_vocab[f"{key}"] = concept[concept['concept_id'].astype(str) == f"{key}"]['concept_code'].values[0]

  return unit_vocab

In [14]:
vocab = get_unit_vocab(['mg/dl', 'umol/L'])

In [15]:
measurement['unit_concept_id'] = measurement['unit_concept_id'].astype(str).replace(vocab)

## 4. Map stays onto measurements

`assign_stays` runs per person: measurements falling inside a stay's start–end range (inclusive) get `stay_no` (1, 2, 3…), `stay_start_date`, `stay_end_date`. Stays with no measurement inside get no number. Outpatient measurements keep `stay_no = NaN`.

Note (pandas ≥ 3): `groupby.apply` no longer passes the grouping column to the function, so `dataset['person_id']` raises KeyError. See `docs/WORKBENCH_INSTRUCTIONS.md` §6a for the one-line wrapper.

In [16]:
def assign_stays(dataset):
  person_stays = stays[stays['person_id'] == dataset['person_id'].values[0]]
  n_visits = len(person_stays)
  stay_counter = 0
  cols_to_take = []
  initial_columns = set(dataset.columns)
  person_stays = person_stays.reset_index(drop = True)
  for visit_idx in range(0, n_visits):
    stay_start_date = pd.to_datetime(person_stays.loc[visit_idx, 'visit_start_date'])
    stay_end_date = pd.to_datetime(person_stays.loc[visit_idx, 'visit_end_date'])
    if len(dataset[(dataset['measurement_date'] >= stay_start_date) & (dataset['measurement_date'] <= stay_end_date)]) != 0:
      stay_rows = dataset[(dataset['measurement_date'] >= stay_start_date) & (dataset['measurement_date'] <= stay_end_date)].copy()
      stay_counter = stay_counter + 1
      stay_rows[f'stay_start_date'] = stay_start_date
      stay_rows[f'stay_end_date'] = stay_end_date
      stay_rows['stay_no'] = stay_counter
      # stay_rows = stay_rows.rename(columns = {'value_as_number':f"creatinine_{stay_counter}"})
      cols = [f'stay_start_date', f'stay_end_date', 'stay_no']
      dataset.loc[stay_rows.index, cols] = stay_rows.loc[:,cols]
    else:
      pass
  #date_cols = list(set(dataset.columns) - initial_columns)
  #date_cols = [x for x in date_cols if 'date' in x]
  #dataset.loc[:,date_cols] = (dataset.loc[:, date_cols])
  dataset = dataset.sort_values(by = ['measurement_date'])
  return dataset

In [17]:
measurement_with_stays = measurement.groupby('person_id', group_keys= False).apply(assign_stays)

/tmp/ipykernel_1532/781271971.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  measurement_with_stays = measurement.groupby('person_id', group_keys= False).apply(assign_stays)


In [18]:
measurement_with_stays = measurement_with_stays.sort_values(['person_id', 'stay_start_date'])

In [19]:
inpatient_creatinine = measurement_with_stays.dropna(subset = ['stay_no'])
outpatient_creatinine = measurement_with_stays.loc[measurement_with_stays['stay_no'].isnull()]

## 5. AKI per stay

`detect_aki` runs per (person, stay). Units first: rows containing `mol` are divided by 88.4, everything is labelled `mg/dL`, rows with `value_as_number > 10` are dropped (eGFR values filed in the creatinine box, data-entry errors). Then the clinical rule:

* **AKI = within the stay, max − min ≥ 0.3 mg/dL OR max ≥ 1.5 × min.** Direction does not matter: a rising series and a falling series (high on admission, low at discharge) both count — KDIGO's retrospective-baseline logic.
* The 48-hour / 7-day windows are **deliberately** not applied; the whole stay is one window.
* Peak (anchor) = date of the highest creatinine in the stay. `months_since_peak` = (measurement date − peak) / 30; 0 at the peak.
* Column names carry the stay number: `2.0_stay_months_since_peak`, `2.0_stay_aki_flag`.

`convert_units` applies the same umol→mg/dL conversion to **outpatient** rows (after the concat, per person). Inpatient rows are already labelled `mg/dL`, so they are not converted twice.

In [20]:
def convert_units(dataset):
  '''
  Converts umol/L rows to mg/dL (per-person group).
  '''
  try:
    rows_to_convert = dataset[dataset['unit_concept_id'].str.contains('mol', case = False)]
    rows_to_convert.loc[:, 'value_as_number'] = rows_to_convert.loc[:,'value_as_number'] / 88.4
    dataset['unit_concept_id'] = 'mg/dL'
    dataset.loc[rows_to_convert.index, 'value_as_number'] = rows_to_convert.loc[:, 'value_as_number']
  except:
    pass
  return dataset

In [21]:
def detect_aki(dataset):
  '''
  Per (person, stay) group: converts umol/L to mg/dL, drops values > 10 mg/dL,
  flags AKI (max - min >= 0.3 mg/dL or max >= 1.5 x min) and writes
  months since the peak creatinine for every row of the stay.
  '''
  try:
    rows_to_convert = dataset[dataset['unit_concept_id'].str.contains('mol', case = False)]
    rows_to_convert.loc[:, 'value_as_number'] = rows_to_convert.loc[:,'value_as_number'] / 88.4
    dataset['unit_concept_id'] = 'mg/dL'
    dataset.loc[rows_to_convert.index, 'value_as_number'] = rows_to_convert.loc[:, 'value_as_number']
  except:
    pass

  dataset = dataset.drop(dataset[dataset['value_as_number'] > 10].index)

  min_cr = pd.to_numeric(dataset['value_as_number']).min()
  max_cr = pd.to_numeric(dataset['value_as_number']).max()

  cr_diff = max_cr - min_cr

  is_aki = int((cr_diff >= 0.3) or (max_cr >= min_cr*1.5))

  if is_aki == 1:
    for date_col in dataset.columns[dataset.columns.str.contains('date', regex = True, case = False)]:
      dataset[date_col] = pd.to_datetime(dataset[date_col], format = 'mixed', errors = 'coerce')
    peak_date = dataset[dataset['value_as_number'] == max_cr]['measurement_date'].values[0]
    dataset['months_since_peak'] = (dataset['measurement_date'] - peak_date)
    dataset[f'{dataset['stay_no'].values[0]}_stay_months_since_peak'] = (dataset['months_since_peak'].dt.days)/30
    dataset['months_since_peak'] = (dataset['months_since_peak'].dt.days)/30
    dataset['aki_flag'] = 1
    dataset[f'{dataset['stay_no'].values[0]}_stay_aki_flag'] = 1
  else:
    dataset['months_since_peak'] = np.nan
    dataset['aki_flag'] = 0
    dataset[f'{dataset['stay_no'].values[0]}_stay_aki_flag'] = 0

  return dataset

## 6. eGFR — CKD-EPI 2021 (race-free)

eGFR = 142 × min(Scr/κ, 1)^α × max(Scr/κ, 1)^−1.200 × 0.9938^age × 1.012 [female]
κ = 0.7 (F) / 0.9 (M); α = −0.241 (F) / −0.302 (M).

Four branches (sex × Scr/κ threshold) computed separately and recombined. Check values: 50 y F, 0.6 → 109.3; 70 F, 1.5 → 37.3; 40 M, 0.8 → 114.7; 60 M, 1.0 → 86.2.
The function also fills NaN in every column containing `flag` with 0 (non-AKI stays and outpatient rows).

In [22]:
def calculate_egfr(dataset, creatinine_col = 'value_as_number'):

  dataset = dataset.reset_index(drop = True)

  # Four branches: sex x creatinine/kappa threshold
  female = dataset[dataset['sex'] == 0].copy()
  male = dataset[dataset['sex'] == 1].copy()

  # Female, Cr <= 0.7
  female_low_cr = female[female[creatinine_col] <= 0.7].copy()
  female_low_cr['egfr'] = female_low_cr[creatinine_col]/0.7
  female_low_cr['egfr'] = female_low_cr['egfr']**(-0.241)
  female_low_age_term = female_low_cr['age_at_measurement'].apply(lambda x: 0.9938**x)
  female_low_cr['egfr'] = 142*female_low_cr['egfr']*female_low_age_term*1.012

  # Female, Cr > 0.7
  female_high_cr = female[female[creatinine_col] > 0.7].copy()
  female_high_cr['egfr'] = female_high_cr[creatinine_col]/0.7
  female_high_cr['egfr'] = female_high_cr['egfr']**(-1.200)
  female_high_age_term = female_high_cr['age_at_measurement'].apply(lambda x: 0.9938**x)
  female_high_cr['egfr'] = 142*female_high_cr['egfr']*female_high_age_term*1.012

  # Male, Cr <= 0.9
  male_low_cr = male[male[creatinine_col] <= 0.9].copy()
  male_low_cr['egfr'] = male_low_cr[creatinine_col]/0.9
  male_low_cr['egfr'] = male_low_cr['egfr']**(-0.302)
  male_low_age_term = male_low_cr['age_at_measurement'].apply(lambda x: 0.9938**x)
  male_low_cr['egfr'] = 142*male_low_cr['egfr']*male_low_age_term

  # Male, Cr > 0.9
  male_high_cr = male[male[creatinine_col] > 0.9].copy()
  male_high_cr['egfr'] = male_high_cr[creatinine_col]/0.9
  male_high_cr['egfr'] = male_high_cr['egfr']**(-1.200)
  male_high_age_term = male_high_cr['age_at_measurement'].apply(lambda x: 0.9938**x)
  male_high_cr['egfr'] = 142*male_high_cr['egfr']*male_high_age_term

  all_egfr = pd.concat([male_high_cr, male_low_cr,female_high_cr, female_low_cr])

  all_egfr = all_egfr.sort_index()

  dataset.loc[all_egfr.index, 'egfr'] = all_egfr.loc[:, 'egfr']

  dataset.loc[:, dataset.columns[dataset.columns.str.contains('flag', case = False, regex = False)]] = dataset.loc[:, dataset.columns[dataset.columns.str.contains('flag', case = False, regex = False)]].fillna(0)

  return dataset

## 7. AKI detection → concat → units → eGFR

In [23]:
measurement_aki = inpatient_creatinine.groupby(['person_id', 'stay_no'],group_keys = False).apply(detect_aki)

/tmp/ipykernel_1532/366534477.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  measurement_aki = inpatient_creatinine.groupby(['person_id', 'stay_no'],group_keys = False).apply(detect_aki)


In [24]:
measurement_aki.head(20)

,measurement_id,person_id,measurement_date,measurement_datetime,value_as_number,measurement_concept_id,unit_concept_id,unit_source_value,sex,birth_year,...,stay_end_date,stay_no,months_since_peak,1.0_stay_months_since_peak,aki_flag,1.0_stay_aki_flag,2.0_stay_months_since_peak,2.0_stay_aki_flag,3.0_stay_months_since_peak,3.0_stay_aki_flag
7,42268752073461757,1028327,2021-05-05,2021-05-05 11:00:00,1.17,3016723,mg/dL,None,0.0,1962,...,2021-05-12,1.0,-0.066667,-0.066667,1,1.0,NaN,NaN,NaN,NaN
8,31623460151100276,1028327,2021-05-06,2021-05-06 09:00:00,1.52,3016723,mg/dL,None,0.0,1962,...,2021-05-12,1.0,-0.033333,-0.033333,1,1.0,NaN,NaN,NaN,NaN
9,8845286263210980,1028327,2021-05-07,2021-05-07 07:00:00,1.59,3016723,mg/dL,None,0.0,1962,...,2021-05-12,1.0,0.000000,0.000000,1,1.0,NaN,NaN,NaN,NaN
10,24368003893757346,1028327,2021-05-08,2021-05-08 09:00:00,1.47,3016723,mg/dL,None,0.0,1962,...,2021-05-12,1.0,0.033333,0.033333,1,1.0,NaN,NaN,NaN,NaN
11,25923198357960754,1028327,2021-05-09,2021-05-09 08:00:00,1.27,3016723,mg/dL,None,0.0,1962,...,2021-05-12,1.0,0.066667,0.066667,1,1.0,NaN,NaN,NaN,NaN
14,44288884639463277,1028327,2021-06-23,2021-06-23 19:56:00,0.77,3016723,mg/dL,None,0.0,1962,...,2021-06-27,2.0,-0.066667,NaN,1,NaN,-0.066667,1.0,NaN,NaN
15,59760887121284531,1028327,2021-06-24,2021-06-24 21:53:00,1.01,3016723,mg/dL,None,0.0,1962,...,2021-06-27,2.0,-0.033333,NaN,1,NaN,-0.033333,1.0,NaN,NaN
16,63700495485301135,1028327,2021-06-25,2021-06-25 14:11:00,1.32,3016723,mg/dL,None,0.0,1962,...,2021-06-27,2.0,0.000000,NaN,1,NaN,0.000000,1.0,NaN,NaN
17,43081016028401587,1028327,2021-06-26,2021-06-26 08:58:00,1.08,3016723,mg/dL,None,0.0,1962,...,2021-06-27,2.0,0.033333,NaN,1,NaN,0.033333,1.0,NaN,NaN
13,52576702493411728,1028327,2021-06-27,2021-06-27 15:56:00,0.92,3016723,mg/dL,None,0.0,1962,...,2021-06-27,2.0,0.066667,NaN,1,NaN,0.066667,1.0,NaN,NaN


In [25]:
all_measurements = pd.concat([measurement_aki, outpatient_creatinine])

In [26]:
all_measurements = all_measurements.groupby('person_id', group_keys= False).apply(convert_units)

/tmp/ipykernel_1532/1307650197.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_measurements = all_measurements.groupby('person_id', group_keys= False).apply(convert_units)


## 8. Months since each AKI peak, for every row

`detect_aki` writes the time column only on that stay's rows. `compute_time_since_each_aki` runs per person and fills the same column on **all** of the person's rows, so outpatient measurements get a negative/positive month value relative to each peak. Stays without AKI have no column; `KeyError/IndexError` are caught and skipped.

In [27]:
def compute_time_since_each_aki(dataset):
  stay_numbers = list(dataset.value_counts('stay_no').index)
  for i in stay_numbers:
    try:
      dataset.loc[dataset[f"{i}_stay_months_since_peak"].isnull(), f"{i}_stay_months_since_peak"] = (pd.to_datetime(dataset.loc[dataset[f"{i}_stay_months_since_peak"].isnull(), 'measurement_date']) - dataset[(dataset['stay_no'] == i) & (dataset['months_since_peak'] == 0)]['measurement_date'].values[0]).apply(lambda x:x.days)/30
    except (KeyError, IndexError):
      pass
  return dataset

## 9. CKD decision per person

`detect_ckd` runs per person. Three branches:

**More than one AKI stay**
* The last AKI stay is the anchor. On the anchor row, the largest value among the other AKI time columns = months between the first and the last AKI.
* If that gap is **> 3 months** → `drop_flag = 1`: separate episodes, the person leaves the single-episode cohort.
* If ≤ 3 months → one episode: baseline and follow-up are computed.

**Exactly one AKI stay** — baseline and follow-up directly.

**No AKI** — `ckd_developed = 0`. (Outside the cohort; filter on `aki_flag == 1`.)

Baseline and follow-up:
* `baseline_egfr` = highest eGFR among all measurements (inpatient + outpatient) in the **365 days before** the peak.
* `followup_egfr` = eGFR of the first measurement, by date, **≥ 3 months** after the peak.
* **CKD = (baseline − follow-up) ≥ 0.3 × baseline OR (baseline > 90 and follow-up < 60).**
* Empty baseline window or no follow-up → `drop_flag = 1`, `ckd_developed = NaN`.

Note: in the single-AKI branch a decided person keeps `drop_flag = NaN`; filter the cohort with `ckd_developed.notna()`.

In [28]:
def detect_ckd(dataset, creatinine_col = 'value_as_number'):
  '''
  Per person: compares baseline eGFR (max within 365 days before the peak)
  with follow-up eGFR (first measurement >= 3 months after the peak)
  and writes ckd_developed / drop_flag. Needs inpatient and outpatient rows together.
  '''
  dataset = dataset.sort_values('measurement_date')
  time_cols = dataset.loc[:,dataset.columns[dataset.columns.str.contains('stay_months_since_peak', regex = False, case= False)]]
  stay_aki_flags = dataset.loc[:,dataset.columns[dataset.columns.str.contains('stay_aki', regex = False, case= False)]]
  if stay_aki_flags.isin([1.0]).any().astype(int).sum() > 1:
    aki_stays = time_cols.dropna(axis = 1).columns.str.strip("_stay_months_since_peak").to_list()
    last_stay = max([float(x) for x in aki_stays])
    max_gap_months = np.max(time_cols[time_cols[f"{last_stay}_stay_months_since_peak"] == 0].dropna(axis = 1).values)
    if max_gap_months <=3:
      dataset['drop_flag'] = 0
      peak_idx = time_cols[time_cols[f"{last_stay}_stay_months_since_peak"] == 0].index[0]
      try:
        followup_egfr = dataset.loc[time_cols[time_cols[f"{last_stay}_stay_months_since_peak"] >= 3].index[0], 'egfr']
        baseline_egfr = np.max(dataset.loc[(dataset['measurement_date'] > dataset.loc[peak_idx, 'measurement_date'] - timedelta(days = 365)) & (dataset['measurement_date'] < dataset.loc[peak_idx, 'measurement_date']), 'egfr'].dropna().values)
        if (baseline_egfr - followup_egfr) >= baseline_egfr*0.3 or (baseline_egfr > 90 and followup_egfr < 60):
          dataset['ckd_developed'] = 1
        else:
          dataset['ckd_developed'] = 0
      except (KeyError, IndexError, ValueError):
        dataset['drop_flag'] = 1
        dataset['ckd_developed'] = np.nan
    elif max_gap_months >3:
      dataset['drop_flag'] = 1
  elif stay_aki_flags.isin([1.0]).any().astype(int).sum() == 1:
    time_cols = time_cols.dropna(axis = 1)
    try:
      peak_idx = dataset.loc[(time_cols == 0).iloc[:,0]].index[0]
      followup_egfr = dataset.loc[time_cols[time_cols >= 3].dropna().index[0], "egfr"]
      baseline_egfr = np.max(dataset.loc[(dataset['measurement_date'] > dataset.loc[peak_idx, 'measurement_date'] - timedelta(days = 365)) & (dataset['measurement_date'] < dataset.loc[peak_idx, 'measurement_date']), 'egfr'].dropna().values)
      if (baseline_egfr - followup_egfr) >= baseline_egfr*0.3 or (baseline_egfr > 90 and followup_egfr < 60):
        dataset['ckd_developed'] = 1
      else:
        dataset['ckd_developed'] = 0
    except (KeyError, IndexError, ValueError):
      dataset['drop_flag'] = 1
      dataset['ckd_developed'] = np.nan
  elif stay_aki_flags.isin([1.0]).any().astype(int).sum() == 0:
    dataset['ckd_developed'] = 0
  return dataset

In [29]:
all_measurements = calculate_egfr(dataset = all_measurements)

In [30]:
all_measurements.head(20)

,measurement_id,person_id,measurement_date,measurement_datetime,value_as_number,measurement_concept_id,unit_concept_id,unit_source_value,sex,birth_year,...,stay_no,months_since_peak,1.0_stay_months_since_peak,aki_flag,1.0_stay_aki_flag,2.0_stay_months_since_peak,2.0_stay_aki_flag,3.0_stay_months_since_peak,3.0_stay_aki_flag,egfr
0,42268752073461757,1028327,2021-05-05,2021-05-05 11:00:00,1.17,3016723,mg/dL,None,0.0,1962,...,1.0,-0.066667,-0.066667,1.0,1.0,NaN,0.0,NaN,0.0,53.753163
1,31623460151100276,1028327,2021-05-06,2021-05-06 09:00:00,1.52,3016723,mg/dL,None,0.0,1962,...,1.0,-0.033333,-0.033333,1.0,1.0,NaN,0.0,NaN,0.0,39.265827
2,8845286263210980,1028327,2021-05-07,2021-05-07 07:00:00,1.59,3016723,mg/dL,None,0.0,1962,...,1.0,0.000000,0.000000,1.0,1.0,NaN,0.0,NaN,0.0,37.200648
3,24368003893757346,1028327,2021-05-08,2021-05-08 09:00:00,1.47,3016723,mg/dL,None,0.0,1962,...,1.0,0.033333,0.033333,1.0,1.0,NaN,0.0,NaN,0.0,40.873917
4,25923198357960754,1028327,2021-05-09,2021-05-09 08:00:00,1.27,3016723,mg/dL,None,0.0,1962,...,1.0,0.066667,0.066667,1.0,1.0,NaN,0.0,NaN,0.0,48.714987
5,44288884639463277,1028327,2021-06-23,2021-06-23 19:56:00,0.77,3016723,mg/dL,None,0.0,1962,...,2.0,-0.066667,NaN,1.0,0.0,-0.066667,1.0,NaN,0.0,88.805157
6,59760887121284531,1028327,2021-06-24,2021-06-24 21:53:00,1.01,3016723,mg/dL,None,0.0,1962,...,2.0,-0.033333,NaN,1.0,0.0,-0.033333,1.0,NaN,0.0,64.127072
7,63700495485301135,1028327,2021-06-25,2021-06-25 14:11:00,1.32,3016723,mg/dL,None,0.0,1962,...,2.0,0.000000,NaN,1.0,0.0,0.000000,1.0,NaN,0.0,46.509143
8,43081016028401587,1028327,2021-06-26,2021-06-26 08:58:00,1.08,3016723,mg/dL,None,0.0,1962,...,2.0,0.033333,NaN,1.0,0.0,0.033333,1.0,NaN,0.0,59.172314
9,52576702493411728,1028327,2021-06-27,2021-06-27 15:56:00,0.92,3016723,mg/dL,None,0.0,1962,...,2.0,0.066667,NaN,1.0,0.0,0.066667,1.0,NaN,0.0,71.726834


In [31]:
all_measurements = all_measurements.groupby('person_id', group_keys = False).apply(compute_time_since_each_aki)

/tmp/ipykernel_1532/3109567233.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  all_measurements = all_measurements.groupby('person_id', group_keys = False).apply(compute_time_since_each_aki)


In [32]:
aki_ckd_table = all_measurements.groupby('person_id', group_keys = False).apply(detect_ckd)

/tmp/ipykernel_1532/1302545048.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  aki_ckd_table = all_measurements.groupby('person_id', group_keys = False).apply(detect_ckd)


In [33]:
aki_ckd_table.head(20)

,measurement_id,person_id,measurement_date,measurement_datetime,value_as_number,measurement_concept_id,unit_concept_id,unit_source_value,sex,birth_year,...,1.0_stay_months_since_peak,aki_flag,1.0_stay_aki_flag,2.0_stay_months_since_peak,2.0_stay_aki_flag,3.0_stay_months_since_peak,3.0_stay_aki_flag,egfr,drop_flag,ckd_developed
1925,45799348140803071,1028327,2019-08-14,2019-08-14 20:21:00,0.84,3016723,mg/dL,None,0.0,1962,...,-21.066667,0.0,0.0,-22.700000,0.0,NaN,0.0,81.001662,0.0,0.0
1927,26723022439739343,1028327,2020-06-19,2020-06-19 19:09:00,0.83,3016723,mg/dL,None,0.0,1962,...,-10.733333,0.0,0.0,-12.366667,0.0,NaN,0.0,81.664697,0.0,0.0
1926,15847105005545475,1028327,2020-06-19,2020-06-19 21:44:00,0.83,3016723,mg/dL,None,0.0,1962,...,-10.733333,0.0,0.0,-12.366667,0.0,NaN,0.0,81.664697,0.0,0.0
1928,50550532022045802,1028327,2020-09-14,2020-09-14 08:27:00,0.76,3016723,mg/dL,None,0.0,1962,...,-7.833333,0.0,0.0,-9.466667,0.0,NaN,0.0,90.771969,0.0,0.0
1929,65319633087819334,1028327,2020-09-23,2020-09-23 13:15:00,0.78,3016723,mg/dL,None,0.0,1962,...,-7.533333,0.0,0.0,-9.166667,0.0,NaN,0.0,87.986196,0.0,0.0
1930,46055679477739872,1028327,2020-09-23,2020-09-23 15:18:00,0.78,3016723,mg/dL,None,0.0,1962,...,-7.533333,0.0,0.0,-9.166667,0.0,NaN,0.0,87.986196,0.0,0.0
1931,12033000598388383,1028327,2021-03-27,2021-03-27 21:59:00,71.60,3020564,mg/dL,None,0.0,1962,...,-1.366667,0.0,0.0,-3.000000,0.0,NaN,0.0,0.385773,0.0,0.0
1932,27701715900819590,1028327,2021-04-20,2021-04-20 14:29:00,83.00,3016723,mg/dL,None,0.0,1962,...,-0.566667,0.0,0.0,-2.200000,0.0,NaN,0.0,0.323097,0.0,0.0
0,42268752073461757,1028327,2021-05-05,2021-05-05 11:00:00,1.17,3016723,mg/dL,None,0.0,1962,...,-0.066667,1.0,1.0,-1.700000,0.0,NaN,0.0,53.753163,0.0,0.0
1,31623460151100276,1028327,2021-05-06,2021-05-06 09:00:00,1.52,3016723,mg/dL,None,0.0,1962,...,-0.033333,1.0,1.0,-1.666667,0.0,NaN,0.0,39.265827,0.0,0.0


## 10. Output and sanity check

`aki_ckd_table`: one row per creatinine measurement. Person-level decision columns (`aki_flag`, `ckd_developed`, `drop_flag`) are written on every row; `groupby('person_id').max()` gives the person table. On the mock CDR (400 persons, 382 with creatinine) this prints: 210 with AKI, 99 dropped, 43 CKD=1, 68 CKD=0.

In [34]:
person_level = aki_ckd_table.groupby('person_id').agg(aki=('aki_flag','max'), ckd=('ckd_developed','max'), drp=('drop_flag','max'))
print("persons:", len(person_level))
print("with AKI:", int((person_level.aki == 1).sum()))
print("dropped:", int((person_level.drp == 1).sum()))
print("CKD=1:", int((person_level.ckd == 1).sum()), "| CKD=0 (with AKI):", int(((person_level.ckd == 0) & (person_level.aki == 1)).sum()))
print("persons with stay_no >= 10:", aki_ckd_table.loc[aki_ckd_table['stay_no'] >= 10, 'person_id'].nunique())
outpatient_rows = aki_ckd_table[aki_ckd_table['stay_no'].isnull()]
print("outpatient rows:", len(outpatient_rows), "| value_as_number > 10:", int((outpatient_rows['value_as_number'] > 10).sum()))

persons: 382
with AKI: 210
dropped: 99
CKD=1: 43 | CKD=0 (with AKI): 68
persons with stay_no >= 10: 0
outpatient rows: 1855 | value_as_number > 10: 56
